# 01 — EDA XAU/USD H1

Exploration du dataset OHLCV avant feature engineering.

Objectifs :
- Statistiques descriptives
- Distribution des log-returns (fat tails, skew)
- Volatility clustering (ACF des rendements² )
- Gaps et jours fériés
- Saisonnalité intra-day

**Prérequis** : `make collect` (ou `python scripts/01_collect_all_data.py --skip-external`) a tourné une fois pour produire `data/interim/xauusd_h1.parquet`.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as ss

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True

df = pd.read_parquet('../data/interim/xauusd_h1.parquet')
print(f'Loaded {len(df):,} rows  |  {df.index.min()} -> {df.index.max()}')
df.head()

## 1. Couverture temporelle et stats descriptives

In [ ]:
from src.data.collect_ohlcv import summary_stats, validate_ohlcv
summary_stats(df)

In [ ]:
df.describe().round(2)

## 2. Évolution du prix

In [ ]:
fig, ax = plt.subplots()
df['close'].plot(ax=ax, lw=0.5, color='darkgoldenrod')
ax.set_title('XAU/USD H1 close — full history')
ax.set_ylabel('USD / oz')
plt.show()

## 3. Log-returns : distribution et fat tails

In [ ]:
r = np.log(df['close']).diff().dropna()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(r, bins=200, color='steelblue', alpha=0.8); axes[0].set_xlim(-0.02, 0.02)
axes[0].set_title('H1 log-returns distribution')
ss.probplot(r.values, dist='norm', plot=axes[1])
axes[1].set_title('QQ-plot vs Normal')
plt.tight_layout(); plt.show()
print(f'mean={r.mean()*1e4:.4f} bp  std={r.std()*1e4:.4f} bp  skew={r.skew():.3f}  kurt={r.kurtosis():.3f}')

## 4. Volatility clustering

In [ ]:
acf_r  = [r.autocorr(lag=k)      for k in range(1, 50)]
acf_r2 = [(r**2).autocorr(lag=k)  for k in range(1, 50)]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, 50), acf_r,  color='gray');     axes[0].axhline(0, color='k', lw=0.5)
axes[0].set_title('ACF — returns (≈ white noise)')
axes[1].bar(range(1, 50), acf_r2, color='firebrick'); axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_title('ACF — squared returns (clustering)')
plt.tight_layout(); plt.show()

## 5. Volatilité réalisée glissante (24h)

In [ ]:
rv = r.rolling(24).std() * np.sqrt(24)
fig, ax = plt.subplots()
rv.plot(ax=ax, lw=0.5, color='firebrick')
ax.set_title('Realized volatility — rolling 24h (× √24)')
plt.show()

## 6. Saisonnalité intra-day

In [ ]:
hourly_vol = r.abs().groupby(r.index.hour).mean() * 100
fig, ax = plt.subplots(figsize=(10, 4))
hourly_vol.plot(kind='bar', ax=ax, color='darkblue', alpha=0.7)
ax.set_title('Average |log-return| by hour-of-day (UTC)')
ax.set_xlabel('Hour'); ax.set_ylabel('avg |log-return| (%)')
plt.show()

## 7. Validation et gaps

In [ ]:
for issue in validate_ohlcv(df):
    print(issue)

In [ ]:
import collections
delta = df.index.to_series().diff().dt.total_seconds().div(3600).dropna()
gap_starts = df.index[1:][delta > 2]
gap_hours  = delta[delta > 2].values
weekday = [(t, h) for t, h in zip(gap_starts, gap_hours)
           if not (((t - pd.Timedelta(hours=h)).weekday() == 4) and t.weekday() in (0, 6))]
print(f'Weekday gaps: {len(weekday)}')
top = collections.Counter((t.month, t.day) for t, _ in weekday).most_common(15)
pd.DataFrame(top, columns=['(month, day)', 'count'])